# Mini Project 5 — E-commerce Event Logs: raw text → silver

**Coaching / interview mode.** I give the tasks; **you write the code yourself**. No solution is provided — if you get stuck, ask and I'll give a hint, not the answer.

**Source file:** `raw_data_5/ecommerce_events.log` — one messy text line per event.
**Goal:** turn this unstructured text into a clean `silver` table (`dev.mini_projects.ecommerce_events_silver`).

**Rules:**
- Each task = a **markdown task** → an **empty code cell** (your code) → **Interview questions** → an answer cell (English).
- I will **not** name the exact method/function — you decide which one fits. The interview questions stay conceptual.
- Work bronze → silver: read raw first, clean step by step, save last. Re-running must be safe (idempotent).

A sample raw line:
```
2022-05-18T10:30:30 | user=U1001 | event=purchase | product=Wireless Mouse | amount=29.99 USD | country=Turkey
18-05-2022 11:05:10 | user=U1002 | event=view | product=Mechanical Keyboard | amount=N/A | country=Germany
```


## Task 1 — Read the raw log (bronze) + drop exact duplicates

**What:** Read the `.log` file so that **each whole line becomes one string column**. Look at the data and the schema. Then remove rows that are **exactly duplicated** lines.
**Why now:** Bronze = raw as-is. You must see the mess before parsing. Some lines are repeated (same event logged twice).
**Rough steps:** read as text (not CSV) → inspect → drop exact duplicate rows → count before/after.

_Path:_ `raw_data_5/ecommerce_events.log` (adjust to your workspace path).


In [ ]:
events_text_df = (
    spark.read
    .format("text")
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/ecommerce_events.log")
)
events_text_df.display()

**Interview questions**
1. Why read this file as *text* instead of CSV? What does the resulting DataFrame look like?
2. Is reading a transformation or an action? When does Spark actually read the file?
3. What's the difference between removing duplicates on *all* columns vs on a *subset* of columns?


1. We load data as txt because it is in the log format and seperators are "|". If we read with csv format spark cuts it from "|" so every cell would be dirty. That's wht we should read this data in the text format. Cells would be like user=1001.
2. Reading is a transformation not an action it is a lazy step. Spark reads the file only when an action runs like display, write, save, count, show.. 
3. If we use subset when we drop duplicate values spark deletes duplicate values for specific columns that we describe with subset. For all columns, spark drops all rows that have same values.


## Task 2 — Extract the fields into columns

**What:** From each line, pull out 6 fields into their own columns: `event_ts_str`, `user_id`, `event_type`, `product`, `amount_str`, `country`. Keep them as **strings** for now (raw extraction).
**Why now:** The line is one blob; analysis needs columns. Each field sits between the `|` separators / after a `key=` label.
**Rough steps:** decide a strategy to pull structured fields out of free text (there is a pattern-matching way and an AI way — pick one, or try both). Produce one column per field.

_Tip:_ look at the separators and the `key=value` shape to design your extraction.


In [ ]:
prompt = """
You will be provided with an e-commerce events  file record.
It is an unstructured text record. Each record represents information
from our events, such as time of event, user identify, type of the event,
which product has bought, amount of spent money for each product and 
which country users come from.

You are asked to parse the log record and extract the following fields:

event_time: the excat timestamp of the event.
user_id: distinct user identity for each event.
event_type: type of the event.
product: each product of the event
amount_str: spent money that for every user.
country: country  for each user.
For each key=value, take the raw text exactly as it appears, up to the next |.
Always return all 6 keys. Do not clean, replace, skip, or merge any value — even if it looks invalid or missing.
Keep every value exactly as it appears in the text. Do not clean, fix or reformat anything.
Only give the final answer in JSON format.
Record:
"""

In [ ]:
from pyspark.sql.functions import concat, col, lit, expr
events_raw_df = (
    events_text_df
    .withColumn("prompt",concat(lit(prompt), col("value")))
    .withColumn("json_extract", expr("""
        ai_query(
            endpoint=> 'databricks-llama-4-maverick',
            request=> prompt,
            responseFormat=> 'struct<extract: struct<
            event_time: string,
            user_id: string,
            event_type: string,
            product: string,
            amount_str: string,
            country: string
            >>'
        )                             
                                     
        """))
)

events_raw_df.display()

We used an LLM, but the LLM does not extract the fields consistently — some rows drop fields or keep the key= labels. We fixed some problems but when we check json_extract column we still have dirty rows ( "country": "country=-", ). Using regexp_extract here is the best way. Regexp_extract is better here because the data has a fixed key=value pattern, and regex is deterministic — it gives the same result every time, unlike the LLM.

In [ ]:
events_text_df.limit(2).display()

Now switching to regexp_extract.

In [ ]:
from pyspark.sql.functions import regexp_extract, col

events_parsed_df = (
    events_text_df                    
    .withColumn("user_id",    regexp_extract(col("value"), "user=([^|]+)", 1))
    .withColumn("event_type", regexp_extract(col("value"), "event=([^|]+)", 1))   
    .withColumn("product",    regexp_extract(col("value"), "product=([^|]+)", 1))   
    .withColumn("amount", regexp_extract(col("value"), "amount=([^|]+)", 1))   
    .withColumn("country",    regexp_extract(col("value"), "country=([^|]+)", 1))  
    .withColumn("events_time", regexp_extract(col("value"), "^([^|]+)", 1)) 
)

events_parsed_df.display()



**Interview questions**
1. What are two different strategies to turn unstructured text into structured columns? Trade-offs of each?
2. After extraction, what data type are all 6 new columns, and why?
3. If a field is missing in a line, what would your extraction return for it?


1. Two strategies: **(a) pattern matching** with `regexp_extract` — I write a pattern for each `key=value` and pull the value out. It is deterministic, fast and repeatable, but I have to know the pattern and it breaks if the format changes. **(b) an LLM** (`ai_query`) — flexible, it reads messy text without a strict pattern, but it is non-deterministic: it can drop fields or keep the `key=` label, and it costs more. For a fixed `key=value` log, regex is the better trade-off.
2. All 6 columns are **strings**, because `regexp_extract` always returns a string. This is bronze/raw extraction — I keep everything as text first and cast to the right types in later steps.
3. If a field is missing, `regexp_extract` returns an **empty string `""`** (not null), because the pattern matches nothing. That is why in a later step I turn placeholders and empty values into real nulls before analysis.

## Task 3 — Parse the event timestamp (careful: TWO formats!)

**What:** Turn `event_ts_str` into a real **timestamp** column `event_ts`.
**Why now:** Downstream date math needs a real timestamp, not text.
**The catch:** the timestamps come in **two different formats** — some like `2022-05-18T10:30:30`, others like `18-05-2022 11:05:10`. A single format string will **not** parse both, and the wrong approach will **crash** on the ones that don't match.
**Rough steps:** find an approach that (a) does **not** throw on a non-matching format and (b) tries more than one format and keeps whichever works. Confirm no valid row became null by accident.


In [ ]:
events_parsed_df.select("events_time").display()

In [ ]:
from pyspark.sql.functions import coalesce,try_to_timestamp,lit, trim

events_parsed_df = events_parsed_df.withColumns({
    "events_time": coalesce(
        try_to_timestamp(trim("events_time") , lit("yyyy-MM-dd'T'HH:mm:ss")),
        try_to_timestamp(trim("events_time") , lit("dd-MM-yyyy HH:mm:ss"))

    )
    
})
events_parsed_df.display()

**Interview questions**
1. If you pass one format string to a strict parser and a row doesn't match, what happens? How is that different from a 'try' parser?
2. How do you handle a column that arrives in multiple date formats?
3. When you pass the format as a plain string, some functions read it as a *column name*. How do you force it to be treated as a literal value?


1. I can use try_to_timestamp instead of to_timestamp because try_to_timestamp  returns null if we don't have that format instead of error. strict parser crashes, try returns null.
2. Instead of using one format I used coalesce to apply multiple formats.
3. When I use coalesce and try_to_timestamp, Spark perceived the format string as a column name. So I used lit function to fix it. Lit function tells Spark "this is a value, not column name" and thanks to it Spark understands the value properly.


In [ ]:
events_parsed_df.display()

## Task 4 — Derive an event date + a human-readable date label

**What:** From `events_time`, create (a) `events_date` as a real **date** type, and (b) `event_date_label` as a **display string** like `18 May 2022`.
**Why now:** Reports group by date; humans read the label. One is for computation, one is for display.
**Rough steps:** derive the date part from the timestamp; separately format the date into the desired text. Notice what data type each result is.


In [ ]:
from pyspark.sql.functions import to_date

events_parsed_df = events_parsed_df.withColumn("events_date", to_date("events_time"))
events_parsed_df.display()

In [ ]:
from pyspark.sql.functions import date_format

events_parsed_df = events_parsed_df.withColumn("events_date_label", date_format("events_date", "dd MMM yyyy"))

events_parsed_df.display()

**Interview questions**
1. What data type does the 'display label' function return? Can you do date math on it? Why / why not?
2. Difference between a `date` and a `timestamp` here?
3. In the format pattern, what's the difference between `MM`, `MMM`, and `MMMM`?


1. It returns string because i used date_format and it returns string value so we cannot do date math on it because it is string not date format.
2. Timestamp represents hours, minutes, seconds but date represents day, month and year.
3. When we use MM PySpark shows us a number like 12, 07.. | when we use MMM PySpark shows us the name of the months like May, Aug (3 letters) | when we use MMMM PySpark shows us the full name of the months like March, April...




In [ ]:
events_parsed_df = events_parsed_df.drop("value")
events_parsed_df.display()


## Task 5 — Date arithmetic: refund deadline + days since event

**What:** Add two columns: `refund_deadline` = **14 days after** `event_date`, and `days_since_event` = number of **days between** a fixed reference date `2022-06-01` and `event_date`.
**Why now:** Business rules live on dates (refund windows, recency).
**Rough steps:** add days to a date for the deadline; compute the day difference between two dates for recency. Mind the argument order for the difference.


In [ ]:
from pyspark.sql.functions import date_add, date_diff, to_date, lit

events_parsed_df = events_parsed_df.withColumns({
    "refund_deadline": date_add("events_date", 14),
    "days_since_event": date_diff(to_date(lit("2022-06-01"), "yyyy-MM-dd"), "events_date")
})

events_parsed_df.display()

**Interview questions**
1. Which function adds *days* vs adds *months*? Why can't you use the day-adder to add months?
2. Does the 'days between' function return a date or a number? What about subtracting two dates directly?
3. If `event_date` is null, what is `refund_deadline`?


1. date_add and month_add functions. If we use date_add instead of add_month to add month it would be deceptive because every months has different number of the days.
2. date_diff returns a number (days count). Subtracting two dates directly returns an interval, not a plain number.
3. If event_date is null, refund_deadline is also null, because any date math on null returns null.




## Task 6 — Turn placeholders into real nulls

**What:** Several columns carry **fake values** that mean 'missing': `amount_str` has `N/A` and `ERROR`; `event_type` has `UNKNOWN`; `country` has `-`. Replace those placeholders with a real **null**.
**Why now:** A null-check won't see `"ERROR"` — it's a filled string. You must first convert placeholders to null so later steps treat them as missing.
**Rough steps:** for each column, when the value is one of the placeholders, produce null; otherwise keep it.


In [ ]:
events_parsed_df.display()

In [ ]:
from pyspark.sql.functions import  col,  when, trim, nullif, lit

events_parsed_df = events_parsed_df.withColumns({
    "amount": when(trim(col("amount")).isin("N/A","ERROR"), None).otherwise(col("amount")),
    "event_type": when(trim(col("event_type")).isin("UNKNOWN"),None).otherwise(col("event_type")),
    "country": nullif(trim("country"),lit("-"))
})

events_parsed_df.display()


In [ ]:
# to control iff we still have invalid values 

events_parsed_df.filter(col("event_type")=="UNKNOWN").display()

In [ ]:
# both of invalid values is made null which is good 

events_parsed_df.filter(col("event_type").isNull()).count()

**Interview questions**
1. Why doesn't a null-check catch the value `"ERROR"`? What's the difference between a placeholder string and a real null?
2. Is turning `"ERROR"` into null the same as *filling* a null (e.g., coalesce)? Explain the direction of each.
3. How would you check, in one line, how many nulls a column has after this step?


1. We should make placeholders null because when we do some calculation for analysis it would be deceptive.
2. No. Turning ERROR into null goes value → null (it empties). Filling a null with coalesce goes null → value (it fills). Opposite directions
3. I would check columns that i applied functions events_parsed_df.filter(col("event_type").isNull()).count()



## Task 7 — Clean then cast the amount to money (decimal)

**What:** Turn `amount_str` into a numeric `amount` with **exact money precision** (2 decimals). The raw text is dirty: `29.99 USD`, `149,90 TL`, `$14.99` — currency words/symbols and a comma used as the decimal separator.
**Why now:** You can't cast dirty text straight to a number. Clean the text **first**, then convert.
**Rough steps:** strip the currency symbols/words, fix the decimal separator, trim spaces → then convert to a precise money type (not a floating type).
**Order matters:** clean text first, cast second.


In [ ]:
from pyspark.sql.functions import regexp_replace

events_parsed_df = events_parsed_df.withColumns({
    "amount": regexp_replace(
    regexp_replace(col("amount"), "[^0-9,.-]", ""),   
    ",", "."                                         
).cast("decimal(10,2)")
})
events_parsed_df.display()

**Interview questions**
1. Why clean the string *before* casting, not after?
2. Why choose an exact money type over a floating type for currency? What goes wrong with the floating type?
3. If you cast dirty text like `"29.99 USD"` directly, what happens?


1. I clean while it is still a string, because casting dirty text fails and returns null — after that there is nothing left to clean.
2. Float stores money approximately, so sums drift with rounding errors. Decimal stores exact values, which money needs.
3. It fails with CAST_INVALID_INPUT (or returns null), because the letters USD cannot be parsed as a number.


## Task 8 — Remove junk characters from product names

**What:** Some `product` values contain junk characters: a backtick (`` ` ``) and a tilde (`~`) — e.g., `Deskt`` + ``op Lamp`, `Noise~Cancel Headset`. Remove those characters so the name is clean.
**Why now:** Dirty text breaks grouping/joins later ('Power~Bank' ≠ 'PowerBank').
**Rough steps:** remove the specific junk characters from the string. (There's a per-character mapping way and a sub-string replace way — either works; try the simplest.)


In [ ]:
events_parsed_df.select("product").display()


In [ ]:
from pyspark.sql.functions import expr

events_parsed_df = events_parsed_df.withColumns({
    "product": expr("replace(replace(product,'`',''),'~','')")
})
events_parsed_df.display()

**Interview questions**
1. Two functions can remove characters from a string — one maps character-by-character, one replaces a sub-string. When would you pick each?
2. One of them needs its search/replace values wrapped as literals or it treats them as column names. Which, and how do you fix it?
3. Is the built-in `functions.replace` the same as `DataFrame.replace`? What's different?


1. translate maps single characters one-by-one (e.g. remove every ` and ~). I pick it to strip a few specific junk characters — short and clean. replace / regexp_replace replaces a whole sub-string or pattern (e.g. "USD" → ""). I pick it when I replace a word or a pattern, not single characters.
2. functions.replace needs its search and replacement values wrapped in lit(), otherwise Spark reads them as column names (INVALID_ATTRIBUTE_NAME). I fix it by wrapping them: replace(col, lit('USD'), lit('')). translate takes plain strings directly, no lit needed. (Inside expr(\"...\") it's SQL, so no lit either.)
3. No, they are different. functions.replace(col, search, replacement) is a column function that replaces a sub-string inside a string column, used in withColumn/select. DataFrame.replace() is a DataFrame method that replaces whole values across the DataFrame (like na.replace), matching the full value, not a sub-string.



## Task 9 — Business rule: drop invalid (negative) amounts

**What:** A negative `amount` is a valid *type* but an invalid *business value* (you can't buy for -19.99). **Remove** the rows where `amount` is negative. Keep everything else — including rows where `amount` is null (missing ≠ invalid).
**Why now:** Silver should hold only business-valid rows, but 'missing' is not the same as 'invalid'.
**Rough steps:** keep the rows you want (amount is null OR amount >= 0); drop the negatives. Watch operator precedence — wrap conditions in parentheses.


In [ ]:
from pyspark.sql.functions import col

events_parsed_df = events_parsed_df.filter((col("amount")>0) | (col("amount").isNull()))
events_parsed_df.display()

**Interview questions**
1. Why keep null amounts but drop negative ones? What's the difference between 'missing' and 'invalid'?
2. Is casting a negative value the fix here, or filtering? Why?
3. Why do combined conditions (`&`, `|`) need parentheses around each side?


1. Null = missing (amount unknown) — we don't know it, that's not an error. Negative = invalid (a real value that breaks a business rule; you can't buy for -19.99). We drop invalid, keep missing, because 'unknown' is not 'wrong'.
2. Filtering. A negative is a valid type (a proper decimal), so casting doesn't fix it — cast is about type, not business rules. The value breaks a rule, so I remove the row with filter.
3. Because & and | have higher precedence than comparisons (>, ==) in Spark/Python. Without parentheses, col>0 | col<5 parses wrong and errors. Parentheses force each comparison to run first.


## Task 10 — Final polish + write silver (idempotent)

**What:** (a) Add `amount_display` = the amount formatted as text with 2 decimals and thousands separator (e.g. `1,149.90`). (b) Rename columns to clean names if needed. (c) Keep only the useful columns. (d) Write the result as a **managed table** `dev.mini_projects.ecommerce_events_silver` so that re-running the notebook **replaces** the table (no duplicates).
**Why now:** Silver is the clean, query-ready output. It must be safe to re-run.
**Rough steps:** format the number for display (note the result type), rename/select, then save with a write mode that overwrites.


In [ ]:
from pyspark.sql.functions import format_number, col

events_clean_df = events_parsed_df.withColumns({ 
    "amount_display": format_number(col("amount"), 2)
})

events_clean_df.display()


In [ ]:
events_clean_df.write.mode("overwrite").saveAsTable("dev.mini_projects.events_silver")

In [ ]:
spark.read.table("dev.mini_projects.events_silver").display()

In [ ]:
silver_df = spark.read.table("dev.mini_projects.events_silver")
silver_df.printSchema()

**Interview questions**
1. The 'format number' step — what type does it return? Can you still sum it afterwards?
2. Which write mode makes the job idempotent (safe to re-run)? What happens with `append` instead?
3. Managed vs external table — what's the practical difference for this silver table?


1. It returns a string. No, I cannot sum it — it is text, not a number. For math I keep the amount decimal column; amount_display is only for showing."
2. overwrite — re-running replaces the table, so the result is always the same, no duplicates. With append, every run adds the rows again → duplicates pile up, so it is not idempotent
3. Managed: Databricks manages both the metadata and the data files; if I drop the table, the data is deleted too. External: I control the data location (a path); dropping the table removes only the metadata, the files stay. For this silver table, managed is simpler — Databricks handles storage. External is for when I need to control the file location or share it with other tools.

---
### When you're done
Run your silver table and show me a few rows + the schema. I'll review like an interviewer:
- did placeholders become real nulls, are amounts a clean decimal, are junk chars gone, timestamps parsed (both formats), negatives dropped, table idempotent?
- I'll push with follow-ups: *why this type? alternative? production concerns?*

Remember: I won't hand you the code. Stuck → ask for a hint.
